# Caption Preprocessing - Flickr8k

Notebook ini mengubah file `captions.txt` mentah jadi vocabulary dan array encoded caption yang siap dipakai untuk training. Jalankan sekali saja, hasilnya disimpan ke folder `features/`.

In [11]:
import os, json, re
import numpy as np
from pathlib import Path


def _find_root(marker="requirements.txt"):
    p = Path(os.getcwd())
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Repo root tidak ketemu, pastikan requirements.txt ada di root.")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
print("Working dir:", REPO_ROOT)

Working dir: c:\Users\Farrel's Laptop\Desktop\mlml\CNN-RNN-digaspolndangakndangak


In [12]:
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))

from shared.caption_utils import (
    clean_caption, build_vocabulary, encode_caption,
    save_vocabulary, load_vocabulary, SPECIAL_TOKENS,
)

## Config

In [13]:
CAPTIONS_TXT   = "data/flickr8k/captions.txt"
FEATURES_DIR   = "features"
IDX_JSON       = os.path.join(FEATURES_DIR, "flickr8k_idx.json")
VOCAB_JSON     = os.path.join(FEATURES_DIR, "vocab.json")
CAPTIONS_NPY   = os.path.join(FEATURES_DIR, "captions_encoded.npy")
SPLITS_JSON    = os.path.join(FEATURES_DIR, "splits.json")

MAX_VOCAB_SIZE = 8000   # ambil 8k kata terbanyak + special tokens
MAX_SEQ_LEN    = 34     # sebagian besar caption di bawah 30 token

## Parse captions.txt

Format file Flickr8k: setiap baris berisi `nama_file,caption`. Setiap gambar punya 5 caption.

In [14]:
caption_map = {}  # image_id -> [caption1, ..., caption5]

with open(CAPTIONS_TXT, "r") as f:
    next(f)  # skip header
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(",", 1)
        if len(parts) != 2:
            continue
        img_file, caption = parts
        img_id = os.path.splitext(img_file)[0]
        caption_map.setdefault(img_id, []).append(caption)

print(f"Total gambar: {len(caption_map)}")
sample_id = list(caption_map.keys())[0]
print(f"Contoh ({sample_id}):", caption_map[sample_id][:2])

Total gambar: 8091
Contoh (1000268201_693b08cb0e): ['A child in a pink dress is climbing up a set of stairs in an entry way .', 'A girl going into a wooden building .']


## Bagi split train / val / test

Split standar Flickr8k: 6000 train, 1000 val, 1000 test. Kita pakai urutan alphabetical dari image ID yang ada di `flickr8k_idx.json`. Kalau punya file split resmi Karpathy, lebih baik pakai itu.

In [15]:
with open(IDX_JSON) as f:
    idx_map = json.load(f)

all_ids   = sorted(idx_map.keys())
train_ids = all_ids[:6000]
val_ids   = all_ids[6000:7000]
test_ids  = all_ids[7000:]

splits = {"train": train_ids, "val": val_ids, "test": test_ids}
with open(SPLITS_JSON, "w") as f:
    json.dump(splits, f)

print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

Train: 6000, Val: 1000, Test: 1091


## Bersihkan caption dan bangun vocabulary

Vocabulary dibangun hanya dari caption training supaya tidak ada data leakage.

In [16]:
train_captions_raw = []
for img_id in train_ids:
    for cap in caption_map.get(img_id, []):
        train_captions_raw.append(clean_caption(cap))

vocab = build_vocabulary(train_captions_raw, max_vocab_size=MAX_VOCAB_SIZE)
save_vocabulary(vocab, VOCAB_JSON)

print(f"Ukuran vocabulary: {len(vocab)}")
print("Special tokens:", {k: v for k, v in vocab.items() if k.startswith('<')})
print("Contoh kata:", list(vocab.items())[4:10])

Ukuran vocabulary: 7684
Special tokens: {'<pad>': 0, '<start>': 1, '<end>': 2, '<unk>': 3}
Contoh kata: [('a', 4), ('in', 5), ('the', 6), ('on', 7), ('is', 8), ('and', 9)]


## Encode semua caption

Untuk setiap gambar, kelima caption-nya di-encode. Output shape: `(jumlah_gambar * 5, MAX_SEQ_LEN)`. Kita simpan juga metadata baris supaya bisa tahu baris mana milik gambar mana.

In [17]:
all_image_ids_ordered = sorted(caption_map.keys())
encoded_rows = []
row_meta     = []  # (image_id, caption_idx)

for img_id in all_image_ids_ordered:
    for cap_idx, cap in enumerate(caption_map[img_id]):
        cleaned = clean_caption(cap)
        encoded = encode_caption(cleaned, vocab, MAX_SEQ_LEN)
        encoded_rows.append(encoded)
        row_meta.append((img_id, cap_idx))

encoded_arr = np.stack(encoded_rows)
np.save(CAPTIONS_NPY, encoded_arr)

meta_path = os.path.join(FEATURES_DIR, "caption_meta.json")
with open(meta_path, "w") as f:
    json.dump(row_meta, f)

print(f"Shape encoded: {encoded_arr.shape}")
print("Contoh baris pertama:", encoded_arr[0])

Shape encoded: (40455, 34)
Contoh baris pertama: [   1    4   44    5    4   97  188    8  127   52    4  414   13  407
    5   30 4568  612    2    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]


## Bangun tf.data.Dataset

Di sini kita definisikan fungsi `build_dataset` yang dipakai oleh notebook training. Dataset menghasilkan tuple `((feature, caption_in), target)`.

- `caption_in`: token yang dimasukkan ke decoder (panjang `MAX_SEQ_LEN`)
- `target`: label next-token, digeser 1 posisi, panjangnya `MAX_SEQ_LEN + 1` karena ada tambahan satu timestep dari feature vector CNN (timestep -1)

Token `<pad>` (index 0) diabaikan oleh loss function saat training.

In [18]:
import tensorflow as tf


def build_dataset(image_ids, shuffle=True, batch_size=64):
    pad_idx = vocab["<pad>"]
    feat_list, cap_in_list, target_list = [], [], []

    for img_id in image_ids:
        feat_idx = idx_map.get(img_id)
        if feat_idx is None:
            continue
        feat = features[feat_idx]
        for cap in caption_map.get(img_id, []):
            cleaned  = clean_caption(cap)
            encoded  = encode_caption(cleaned, vocab, MAX_SEQ_LEN)
            target   = np.append(encoded[1:], [pad_idx, pad_idx])  # (MAX_SEQ_LEN+1,)
            feat_list.append(feat)
            cap_in_list.append(encoded)
            target_list.append(target)

    ds = tf.data.Dataset.from_tensor_slices((
        (np.array(feat_list,   dtype=np.float32),
         np.array(cap_in_list, dtype=np.int32)),
        np.array(target_list,  dtype=np.int32),
    ))
    if shuffle:
        ds = ds.shuffle(len(feat_list), reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

## Preview dataset

In [19]:
features = np.load(os.path.join(FEATURES_DIR, "flickr8k_features.npy"))

train_ds = build_dataset(train_ids, shuffle=True,  batch_size=4)
val_ds   = build_dataset(val_ids,   shuffle=False, batch_size=4)

for (feat_batch, cap_batch), tgt_batch in train_ds.take(1):
    print("feature batch  :", feat_batch.shape)    # (4, 2048)
    print("caption_in     :", cap_batch.shape)     # (4, MAX_SEQ_LEN)
    print("target         :", tgt_batch.shape)     # (4, MAX_SEQ_LEN+1)

feature batch  : (4, 2048)
caption_in     : (4, 34)
target         : (4, 35)


## Cek decoding token

In [20]:
idx2word = {v: k for k, v in vocab.items()}

for (_, cap_batch), tgt_batch in train_ds.take(1):
    sample_in  = cap_batch[0].numpy()
    sample_tgt = tgt_batch[0].numpy()
    print("caption_in :", " ".join(idx2word.get(int(t), "?") for t in sample_in))
    print("target     :", " ".join(idx2word.get(int(t), "?") for t in sample_tgt))
    break

caption_in : <start> children jump on a trampoline <end> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
target     : children jump on a trampoline <end> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
